# 06 — State representation research: the Fatimah version

Now take everything from Elder Mist and translate it into the research problem:

> Raw sources come in. What state representation should the harness receive so it can choose good actions?


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">The research pipeline</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">RAW SOURCES</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">STATE REPRESENTATION</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">HARNESS / POLICY</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">ACTION</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">OUTCOME</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))

from eldermist import ElderMistReplicaEnv
from eldermist.qlearning import train_q, q_policy
from visuals import pipeline

fig = pipeline(
    ["raw sources","candidate state","same learner","actions","held-out outcome"],
    title="What we vary vs what we hold fixed"
)
plt.show()

## 1. Telemetry is not automatically state representation

Raw inputs might be screenshots, accessibility trees, mouse/keyboard traces, messages, history, or app data.

A representation is the transformation that decides what decision-relevant information survives.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Raw observations → representation</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SCREEN</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">A11Y TREE</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">HISTORY</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">TRANSFORM</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">STATE zₜ</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
sources = ["screenshot","a11y tree","history","cursor/keys","app metadata"]
raw_size = [100,35,60,20,15]

plt.figure(figsize=(9,4))
plt.bar(sources, raw_size)
plt.ylabel("Illustrative raw information volume")
plt.title("More raw data does not automatically mean a better state")
plt.xticks(rotation=20)
plt.show()

## 2. Candidate representations

In Elder Mist we can compare FULL, NO_SWORD, NO_LOCKS, and MINIMAL.

In computer use, the analogues might be screenshot-only, tree-only, screenshot+tree, or a compact inferred task state.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Elder Mist candidate states</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">FULL</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">NO SWORD</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">NO LOCKS</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">MINIMAL</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
reps = ["full","no_sword","no_locks","minimal"]
envs = {}
sizes = []

for rep in reps:
    e = ElderMistReplicaEnv(representation=rep)
    e.reset(seed=1)
    envs[rep] = e
    sizes.append(len(e.compact_state()))

plt.figure(figsize=(8,4))
plt.bar(reps, sizes)
plt.ylabel("Tuple length")
plt.title("How much information each Elder Mist representation retains")
plt.show()

## 3. Fair comparison means holding the rest fixed

If you change the model, task set, training budget, and representation all at once, you cannot tell what caused the result.

So hold constant:

- environment
- task distribution
- learner
- reward
- training budget
- evaluation seeds

Change only the representation.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Controlled experiment</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SAME TASKS</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">SAME LEARNER</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">ONLY REP CHANGES</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">HELD-OUT TEST</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
train_episodes = 4000
eval_seeds = range(50000, 50120)
rows = []
models = {}

for rep in reps:
    q, _ = train_q(
        lambda rep=rep: ElderMistReplicaEnv(representation=rep, reward_mode="shaped"),
        episodes=train_episodes,
        seed=123
    )
    models[rep] = q

    wins, turns, counters = [], [], []
    for seed in eval_seeds:
        e = ElderMistReplicaEnv(representation=rep, reward_mode="shaped")
        o,_ = e.reset(seed=seed)
        while True:
            a = q_policy(q, e)
            o,r,term,trunc,_ = e.step(a)
            if term or trunc: break
        wins.append(int(e.state.boss_hp <= 0))
        turns.append(e.state.turn)
        counters.append(e.state.counter_slashes)

    rows.append({
        "representation": rep,
        "win_rate": np.mean(wins),
        "avg_turns": np.mean(turns),
        "avg_counters": np.mean(counters),
        "fields": len(envs[rep].compact_state())
    })

results = pd.DataFrame(rows)
display(results)

plt.figure(figsize=(8,4))
plt.bar(results["representation"], results["win_rate"])
plt.ylim(0,1)
plt.ylabel("Held-out win rate")
plt.title("Downstream performance is the test of representation quality")
plt.show()

## 4. Why a representation can fail

If Sword-active and Sword-down worlds map to the same representation, the policy cannot choose differently between them.

This is **state aliasing**: strategically different situations collapse into the same input.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Aliasing destroys useful distinctions</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">WORLD A</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">SAME zₜ</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">WORLD B</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">POLICY CAN&#x27;T DISTINGUISH</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
alias_risk = {
    "full": 0.05,
    "no_sword": 0.75,
    "no_locks": 0.55,
    "minimal": 0.90
}

plt.figure(figsize=(8,4))
plt.bar(alias_risk.keys(), alias_risk.values())
plt.ylim(0,1)
plt.ylabel("Illustrative aliasing risk")
plt.title("Removing variables can merge action-relevant worlds")
plt.show()

## 5. “Best” is usually multi-objective

A huge state may improve performance but cost more tokens, latency, memory, or inference time.

So compare performance **and** cost.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">No single metric is always enough</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SUCCESS</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">TOKENS</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">LATENCY</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">TOOL ERRORS</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">PARETO FRONTIER</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(results["fields"], results["win_rate"], s=120)
for _, r in results.iterrows():
    plt.text(r["fields"]+0.1, r["win_rate"], r["representation"])
plt.xlabel("Representation size / cost proxy")
plt.ylabel("Held-out win rate")
plt.ylim(0,1)
plt.title("Performance vs representation cost")
plt.show()

## 6. Map Elder Mist directly to screenshot + accessibility tree

The structure of the experiment is identical.

What changes is the source data and the action space.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Observable Intuition / computer-use analogue</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SCREEN + TREE + HISTORY</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">zₜ</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">HARNESS π</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">TOOL/UI ACTION</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">TASK SUCCESS</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
candidates = ["screen","tree","screen+tree","screen+tree+history","structured state"]
illustrative_cost = [90,35,110,150,25]
illustrative_success = [0.58,0.63,0.76,0.82,0.80]

plt.figure(figsize=(8,5))
plt.scatter(illustrative_cost, illustrative_success, s=120)
for x,y,n in zip(illustrative_cost,illustrative_success,candidates):
    plt.text(x+2,y,n)
plt.xlabel("Illustrative representation cost")
plt.ylabel("Illustrative task success")
plt.ylim(0.45,0.9)
plt.title("What a computer-use state-representation study could look like")
plt.show()

print("Illustrative only: the point is the experimental design, not these fake numbers.")

## 7. What does “train the harness” actually mean?

You need to identify the parameterized component.

Possibilities include state encoder, retrieval/memory selector, planner, LLM/LoRA, tool-selection policy, or some combination.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Which component actually learns?</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">STATE ENCODER</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">MEMORY</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">PLANNER</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">LLM</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">TOOL POLICY</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
components = ["state encoder","retrieval","planner","LLM/LoRA","tool policy"]
trainable_question = [1,1,1,1,1]

plt.figure(figsize=(9,4))
plt.bar(components, trainable_question)
plt.ylim(0,1.4)
plt.ylabel("Potential trainable component")
plt.xticks(rotation=20)
plt.title("“Train the harness” is incomplete until you name what parameters change")
plt.show()

## 8. Why τ² helps

τ² gives a controlled interactive environment: observations, actions, world transitions, and machine-checkable outcomes.

That makes it useful as a test bed for representation ablations before moving to messier real telemetry.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Why a benchmark environment matters</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">TASK</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">OBSERVE</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">ACT</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">WORLD CHANGES</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">ASSERT SUCCESS</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
stages = ["raw sources","representation","policy","action","transition","reward"]
difficulty = [3,5,4,3,4,2]

plt.figure(figsize=(9,4))
plt.plot(stages, difficulty, marker="o")
plt.ylabel("Where research complexity can live")
plt.title("τ² controls the loop so you can isolate representation questions")
plt.xticks(rotation=20)
plt.show()

## 9. Questions to bring into the meeting

1. What are the raw sources?
2. What is one action?
3. What is one episode?
4. What is hidden?
5. What is reward?
6. Which component learns?
7. How are representations compared fairly?
8. Imitation or outcome optimization?
9. Where is uncertainty represented?
10. What is the smallest sufficient representation?


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Turn vague research into explicit variables</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SOURCES?</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">STATE?</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">ACTION?</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">REWARD?</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">LEARNER?</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
questions = [
    "raw sources","action space","episode","hidden state","reward",
    "trainable component","fair eval","imitation vs outcome","uncertainty","sufficiency"
]
importance = [10,9,9,10,10,10,9,9,8,10]

plt.figure(figsize=(10,5))
plt.barh(questions, importance)
plt.xlim(0,11)
plt.xlabel("Ask this explicitly")
plt.title("Meeting checklist")
plt.show()

## Final sentence to memorize

**State-representation research asks what information the policy must retain so that different future consequences and good actions remain distinguishable.**

You now have the toy experiment, the POMDP extension, the neural-weight extension, and the direct computer-use mapping.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">The whole point</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">RAW DATA</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">KEEP WHAT MATTERS</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">POLICY CAN DISTINGUISH</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">BETTER ACTIONS</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>
